# AutoGluon

In [1]:
import qlib
import pandas as pd
import numpy as np
from qlib.contrib.data.handler import Alpha158
from qlib.data.dataset import DatasetH
from autogluon.tabular import TabularPredictor

In [2]:
# 1. Inicializar Qlib
qlib.init(provider_uri='/home/toni/.qlib/qlib_data/us_data',region=qlib.constant.REG_US)

[12014:MainThread](2026-05-06 01:33:23,048) INFO - qlib.Initialization - [config.py:453] - default_conf: client.
[12014:MainThread](2026-05-06 01:33:24,035) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[12014:MainThread](2026-05-06 01:33:24,037) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/home/toni/.qlib/qlib_data/us_data')}


In [3]:
import qlib
from qlib.constant import REG_US
from qlib.data import D

qlib.init(provider_uri="/home/toni/.qlib/qlib_data/us_data", region=REG_US)

df = D.features(["^GSPC"], ["$close", "$factor"], start_time="2020-01-01", end_time="2026-05-01", freq="day")

print(df.shape)
print(df.empty)
print(df.tail())


[12014:MainThread](2026-05-06 01:33:24,056) INFO - qlib.Initialization - [config.py:453] - default_conf: client.
[12014:MainThread](2026-05-06 01:33:24,059) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[12014:MainThread](2026-05-06 01:33:24,061) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/home/toni/.qlib/qlib_data/us_data')}


(1584, 2)
False
                         $close   $factor
instrument datetime                      
^GSPC      2026-04-16  4.792431  0.000681
           2026-04-17  4.850134  0.000681
           2026-04-20  4.838618  0.000681
           2026-04-21  4.807902  0.000681
           2026-04-22  4.858193  0.000681


In [4]:
# 2. Cargar dataset Alpha154/158
data_handler_config= {
    "start_time":"2018-01-01",
    "end_time":"2026-04-01",
    "fit_start_time":"2018-01-01",
    "fit_end_time":"2024-12-31",
    "instruments":"sp500",
}

handler= Alpha158(**data_handler_config)

[12014:MainThread](2026-05-06 01:33:38,600) INFO - qlib.timer - [log.py:127] - Time cost: 14.336s | Loading data Done
[12014:MainThread](2026-05-06 01:33:39,637) INFO - qlib.timer - [log.py:127] - Time cost: 0.225s | DropnaLabel Done
[12014:MainThread](2026-05-06 01:33:41,101) INFO - qlib.timer - [log.py:127] - Time cost: 1.462s | CSZScoreNorm Done
[12014:MainThread](2026-05-06 01:33:41,105) INFO - qlib.timer - [log.py:127] - Time cost: 2.503s | fit & process data Done
[12014:MainThread](2026-05-06 01:33:41,107) INFO - qlib.timer - [log.py:127] - Time cost: 16.844s | Init data Done


In [5]:
# 3. Obtener features y labels
features_df= handler.fetch(col_set="feature")
labels_df= handler.fetch(col_set="label")

In [6]:
label_column='LABEL0'

In [7]:
# 4. Combinar en un solo DataFrame
data= pd.concat([features_df, labels_df],axis=1)
# Remove columns that are completely unavailable, e.g. VWAP0 if no $vwap exists
data = data.dropna(axis=1, how="all")

# Make sure label is numeric and finite
data[label_column] = pd.to_numeric(data[label_column], errors="coerce")
data = data.replace([np.inf, -np.inf], np.nan)
data = data[np.isfinite(data[label_column])]

print("Rows after label cleanup:", len(data))
print("Remaining label NaNs:", data[label_column].isna().sum())

Rows after label cleanup: 988748
Remaining label NaNs: 0


In [8]:
data

KMID      KLEN     KMID2       KUP      KUP2  \
datetime   instrument                                                     
2018-01-02 A           0.002670  0.008158  0.327275  0.004301  0.527273   
           AAL         0.012612  0.022931  0.550001  0.002102  0.091664   
           AAP         0.051437  0.081467  0.631385  0.018236  0.223845   
           AAPL        0.012341  0.017866  0.690785  0.000235  0.013159   
           ABBV        0.013074  0.022133  0.590700  0.005044  0.227905   
...                         ...       ...       ...       ...       ...   
2026-04-01 XYZ        -0.026647  0.032287 -0.825313  0.004169  0.129117   
           YUM        -0.022202  0.029137 -0.762007  0.000000  0.000000   
           ZBH         0.005634  0.010826  0.520410  0.001436  0.132655   
           ZBRA       -0.007802  0.026183 -0.297989  0.006127  0.234003   
           ZTS        -0.007699  0.018020 -0.427226  0.005838  0.323946   

                           KLOW     KLOW2      KSFT     KSFT2     OPEN0  ...  \
datetime   instrument                                                    ...   
2018-01-02 A           0.001187  0.145452 -0.000445 -0.054546  0.997337  ...   
           AAL         0.008217  0.358336  0.018727  0.816673  0.987545  ...   
           AAP         0.011794  0.144769  0.044995  0.552309  0.951079  ...   
           AAPL        0.005289  0.296055  0.017395  0.973681  0.987809  ...   
           ABBV        0.004015  0.181396  0.012045  0.544191  0.987095  ...   
...                         ...       ...       ...       ...       ...  ...   
2026-04-01 XYZ         0.001471  0.045570 -0.029344 -0.908861  1.027376  ...   
           YUM         0.006934  0.237993 -0.015268 -0.524014  1.022707  ...   
           ZBH         0.003756  0.346934  0.007954  0.734689  0.994397  ...   
           ZBRA        0.012254  0.468008 -0.001675 -0.063983  1.007864  ...   
           ZTS         0.004484  0.248828 -0.009052 -0.502344  1.007758  ...   

                        VSUMN10   VSUMN20   VSUMN30   VSUMN60    VSUMD5  \
datetime   instrument                                                     
2018-01-02 A           0.726731  0.586454  0.532266  0.515107 -0.083773   
           AAL         0.634902  0.533584  0.504173  0.500709  0.456964   
           AAP         0.403379  0.482207  0.490406  0.494112  0.388357   
           AAPL        0.608529  0.550233  0.495184  0.494970  0.211619   
           ABBV        0.737471  0.504350  0.497144  0.499255  0.472006   
...                         ...       ...       ...       ...       ...   
2026-04-01 XYZ         0.620792  0.567859  0.512933  0.507910  0.166319   
           YUM         0.527585  0.510032  0.511144  0.520642 -0.029544   
           ZBH         0.462439  0.453934  0.497771  0.492654  0.415168   
           ZBRA        0.485893  0.516755  0.516473  0.504114  0.435889   
           ZTS         0.574554  0.564000  0.531321  0.518275 -0.186424   

                        VSUMD10   VSUMD20   VSUMD30   VSUMD60    LABEL0  
datetime   instrument                                                    
2018-01-02 A          -0.453462 -0.172908 -0.064533 -0.030214 -0.007501  
           AAL        -0.269803 -0.067167 -0.008345 -0.001418  0.006305  
           AAP         0.193242  0.035586  0.019188  0.011776  0.036899  
           AAPL       -0.217058 -0.100466  0.009631  0.010061  0.004645  
           ABBV       -0.474942 -0.008700  0.005711  0.001491 -0.005703  
...                         ...       ...       ...       ...       ...  
2026-04-01 XYZ        -0.241585 -0.135718 -0.025867 -0.015820  0.015055  
           YUM        -0.055170 -0.020064 -0.022288 -0.041285  0.008136  
           ZBH         0.075121  0.092132  0.004458  0.014693  0.001210  
           ZBRA        0.028214 -0.033509 -0.032946 -0.008228  0.040820  
           ZTS        -0.149107 -0.127999 -0.062642 -0.036550  0.002713  

[988748 rows x 158 columns]

In [9]:
# 5. Preparar train/test split temporal (importante en finanzas!)
dt = data.index.get_level_values("datetime")
train_data = data[dt < "2023-01-01"]
test_data = data[dt >= "2023-01-01"]

train_data = train_data.reset_index()
test_data = test_data.reset_index()

print(f"Train samples:{len(train_data)}, Test samples:{len(test_data)}")
print(f"Features:{features_df.shape[1]}")

Train samples:585539, Test samples:403209
Features:158


In [10]:
# 6. Entrenar con AutoGluon
label_column='LABEL0'# Nombre típico de la columna objetivo en Qlib

predictor= TabularPredictor(
    label=label_column,
    path='qlib_autogluon_models/',
    eval_metric='rmse'# o 'mae' para regresión
).fit(
    train_data=train_data,
    time_limit=3600,# 1 hora
    presets='best_quality',
    verbosity=2
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          16
Pytorch Version:    2.5.1
CUDA Version:       12.4
GPU Memory:         GPU 0: 4.00/4.00 GB
Total GPU Memory:   Free: 4.00 GB, Allocated: 0.00 GB, Total: 4.00 GB
GPU Count:          1
Memory Avail:       10.35 GB / 15.51 GB (66.8%)
Disk Space Avail:   12.81 GB / 466.57 GB (2.7%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by s

In [11]:
# 7. Evaluar
test_features= test_data.drop(columns=[label_column])
test_labels= test_data[label_column]

predictions= predictor.predict(test_features)
performance= predictor.evaluate(test_data)

print(f"\n📈 Performance en Test:")
print(performance)

# 8. Leaderboard de modelos
predictor.leaderboard(test_data)


📈 Performance en Test:
{'root_mean_squared_error': np.float32(-0.022435205), 'mean_squared_error': -0.0005033384077250957, 'mean_absolute_error': -0.015804260969161987, 'r2': -0.28082358837127686, 'pearsonr': 0.0006956600700505078, 'median_absolute_error': -0.011878792196512222}


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBM_BAG_L1,-0.021173,-0.018873,root_mean_squared_error,17.721047,5.620235,234.716693,17.721047,5.620235,234.716693,1,True,2
1,WeightedEnsemble_L2,-0.021372,-0.018557,root_mean_squared_error,248.115592,120.786489,1708.406621,0.010917,0.004452,0.209590,2,True,3
2,LightGBMXT_BAG_L1,-0.021828,-0.018662,root_mean_squared_error,230.383628,115.161803,1473.480338,230.383628,115.161803,1473.480338,1,True,1
3,CatBoost_BAG_L2,-0.021886,-0.018553,root_mean_squared_error,248.892997,121.263100,1766.675648,0.788321,0.481062,58.478618,2,True,6
4,LightGBMXT_BAG_L2,-0.022396,-0.018168,root_mean_squared_error,265.081784,124.266092,1955.487857,16.977108,3.484054,247.290826,2,True,4
5,WeightedEnsemble_L3,-0.022435,-0.018121,root_mean_squared_error,269.364012,125.018734,2047.045054,0.024404,0.007489,0.507330,3,True,7
6,LightGBM_BAG_L2,-0.022504,-0.018156,root_mean_squared_error,252.362500,121.527191,1799.246898,4.257825,0.745153,91.049867,2,True,5
